# 💊 FingerWave for drug discovery: fast electrostatics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pavankrovvidi/pavan-krovvidi-fingerwave/blob/main/examples/drug_discovery/FingerWave_Drug_Discovery.ipynb)

Whether a drug binds to a protein depends heavily on the **electric field** around the molecules. Physics solvers compute it
with the Poisson–Boltzmann equation, which is slow when millions of candidates must be screened.

Its core term **∇·(ε∇φ)** has the same form as the FingerWave wave layer **∇·(D∇Ψ)**. Atom charges act as *seeds* and the
molecule's shape steers the wave. This notebook compares the **Pavan Krovvidi FingerWave Neural Network** with a same-size
U-Net on 2D synthetic molecules whose exact field comes from a physics solver.

Created by **Pavan Kumar Krovvidi** · [GitHub](https://github.com/Pavankrovvidi/pavan-krovvidi-fingerwave) · Apache 2.0

## 1. Install and download the demo files

In [ ]:
!pip install -q "pavan-krovvidi-fingerwave[viz]" scipy
!wget -q -N https://raw.githubusercontent.com/Pavankrovvidi/pavan-krovvidi-fingerwave/main/examples/drug_discovery/electro_data.py https://raw.githubusercontent.com/Pavankrovvidi/pavan-krovvidi-fingerwave/main/examples/drug_discovery/electro_model.py https://raw.githubusercontent.com/Pavankrovvidi/pavan-krovvidi-fingerwave/main/examples/drug_discovery/fingerwave_electro.pt https://raw.githubusercontent.com/Pavankrovvidi/pavan-krovvidi-fingerwave/main/examples/drug_discovery/unet_electro.pt

## 2. Load the two trained models

In [ ]:
import time, numpy as np, torch, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from electro_data import make_dataset, molecule, fields, solve
from electro_model import FingerWaveElectro, UNet, to_tensors, metrics
fw = FingerWaveElectro(); fw.load_state_dict(torch.load("fingerwave_electro.pt")); fw.eval()
un = UNet(4, 1, 16);      un.load_state_dict(torch.load("unet_electro.pt"));      un.eval()
print("parameters  FingerWave:", sum(p.numel() for p in fw.parameters()), " U-Net:", sum(p.numel() for p in un.parameters()))

## 3. 100 new molecules: who is closer to the physics solver?

In [ ]:
t = time.time(); test = make_dataset(100, seed=11); solver_ms = (time.time() - t) / 100 * 1000
X, Y, S = to_tensors(test)
with torch.no_grad():
    t = time.time(); pf = fw(X); fw_ms = (time.time() - t) / 100 * 1000
    pu = un(X)
for name, p in [("FingerWave", pf), ("U-Net", pu)]:
    print(f"{name:>10}:", {k: round(v, 4) for k, v in metrics(p, Y, S).items()})
err = lambda p: ((p - Y).flatten(1).norm(dim=1) / Y.flatten(1).norm(dim=1))
print(f"FingerWave closer than U-Net on {(err(pf) < err(pu)).float().mean()*100:.0f}% of molecules")
print(f"time per molecule: solver {solver_ms:.1f} ms (incl. generation) | FingerWave {fw_ms:.1f} ms")

## 4. 🧪 Design your own molecule

Edit the atoms (x, y in a 64 × 64 box, radius, charge) and run the cell.

In [ ]:
ATOMS = [  # x,  y,  radius, charge
    (26, 28, 3.0, +1.0),
    (30, 31, 3.0, -0.5),
    (34, 29, 3.2, +0.8),
    (38, 33, 2.8, -1.0),
    (33, 36, 3.0, +0.5),
]
pos = np.array([[a[0], a[1]] for a in ATOMS], float); rad = np.array([a[2] for a in ATOMS]); q = np.array([a[3] for a in ATOMS])
eps, kap, rho, inside, surf = fields(pos, rad, q)
d = {"eps": eps[None], "kap": kap[None], "rho": rho[None], "inside": inside[None], "surface": surf[None], "phi": solve(eps, kap, rho)[None]}
d = {k: v.astype(np.float32) for k, v in d.items()}
x, y, s = to_tensors(d)
with torch.no_grad(): f, conf = fw(x, return_confidence=True); u = un(x)
v = float(y.abs().max()); nrm = TwoSlopeNorm(0, -v, v)
fig, ax = plt.subplots(1, 5, figsize=(17, 3.6))
ax[0].imshow(inside, cmap="Greys"); ax[0].scatter(pos[:, 0], pos[:, 1], c=q, cmap="RdBu_r", vmin=-1, vmax=1, s=60, edgecolors="k"); ax[0].set_title("your molecule")
for a, img, t in [(ax[1], y, "physics solver"), (ax[2], f, "FingerWave"), (ax[3], u, "U-Net")]:
    a.imshow(img[0, 0], cmap="RdBu_r", norm=nrm); a.set_title(t if t == "physics solver" else f"{t}  error {((img - y).norm() / y.norm()).item()*100:.1f}%")
ax[4].imshow(conf[0, 0], cmap="magma", vmin=0, vmax=1); ax[4].set_title("FingerWave confidence")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Notes

* Pilot setting: 2D synthetic molecules on a 64 × 64 grid, linearized Poisson–Boltzmann with ε = 2 inside and 80 outside.
* Benchmark (3 seeds, 300 unseen molecules): field error **17.3 %** (FingerWave) vs **18.8 %** (U-Net); surface error 7 % lower.
* Next: 3D protein–ligand complexes and comparison with industry solvers such as APBS.

⭐ Star the repo · 💬 tell me what you find on [LinkedIn](https://www.linkedin.com/in/pavan-kumar-krovvidi-927290250)